In [3]:
import os
import dotenv
from huggingface_hub import hf_hub_download
dotenv.load_dotenv()
HUGGING_FACE_API_KEY = os.environ.get("HUGGING_FACE_API_KEY")

In [4]:
model_name = "meta-llama/Llama-3.2-1B-Instruct"

In [40]:
model_name = "microsoft/Phi-4-mini-instruct"

## Download model ##

In [3]:
from transformers import AutoTokenizer, AutoModelForCausalLM
tokenizer = AutoTokenizer.from_pretrained(model_name, token=HUGGING_FACE_API_KEY)
model = AutoModelForCausalLM.from_pretrained(model_name, token=HUGGING_FACE_API_KEY)


c:\unimelb\2025 sem 1\nlp\as3\NLP_PRJ\venv\Lib\site-packages\huggingface_hub\file_download.py:144: UserWarning: `huggingface_hub` cache-system uses symlinks by default to efficiently store duplicated files but your machine does not support them in C:\Users\even0\.cache\huggingface\hub\models--meta-llama--Llama-3.2-1B-Instruct. Caching files will still work but in a degraded version that might require more space on your disk. This warning can be disabled by setting the `HF_HUB_DISABLE_SYMLINKS_WARNING` environment variable. For more details, see https://huggingface.co/docs/huggingface_hub/how-to-cache#limitations.
To support symlinks on Windows, you either need to activate Developer Mode or to run Python as an administrator. In order to activate developer mode, see this article: https://docs.microsoft.com/en-us/windows/apps/get-started/enable-your-device-for-development
  warnings.warn(message)


## Save model ##

In [4]:
tokenizer.save_pretrained(f"tokenizers/{model_name}")
model.save_pretrained(f"models/{model_name}")



## Use Model ##

In [2]:
import torch

# Make sure CUDA is available
assert torch.cuda.is_available(), "CUDA is not available!"


In [41]:
from transformers import AutoTokenizer, AutoModelForCausalLM
tokenizer = AutoTokenizer.from_pretrained(f"tokenizers/{model_name}")
model = AutoModelForCausalLM.from_pretrained(f"models/{model_name}", torch_dtype=torch.float16, device_map="cuda").to("cuda")



Loading checkpoint shards: 100%|██████████| 4/4 [00:47<00:00, 11.99s/it]


## Load Data ##

In [ ]:
import os, json, pandas as pd

DATA_DIR   = "../data"
TRAIN_FILE = os.path.join(DATA_DIR, "train-claims.json")
DEV_FILE   = os.path.join(DATA_DIR, "dev-claims.json")
TEST_FILE  = os.path.join(DATA_DIR, "test-claims-unlabelled.json")
EVID_FILE  = os.path.join(DATA_DIR, "evidence.json")


def load_claims(path: str, labelled: bool = True) -> pd.DataFrame:
   
    with open(path, "r", encoding="utf-8") as f:
        raw = json.load(f)

    rows = []
    for cid, info in raw.items():
        row = {
            "claim_id":   cid,
            "claim_text": info.get("claim_text", "")
        }
        if labelled:
            row["label"]      = info["claim_label"]
            row["evid_ids"]   = info["evidences"]     # list[str]
        rows.append(row)

    df = pd.DataFrame(rows)

   
    if labelled:
        df["label"] = df["label"].astype("category")

    return df


def load_evidence(path: str):
    
    with open(path, "r", encoding="utf-8") as f:
        raw = json.load(f)

    df   = pd.DataFrame([{"evid_id": k, "evid_text": v} for k, v in raw.items()])
    edict = {k: v for k, v in raw.items()}
    return df, edict



df_train = load_claims(TRAIN_FILE, labelled=True)
df_dev   = load_claims(DEV_FILE,   labelled=True)
df_test  = load_claims(TEST_FILE,  labelled=False)

df_evid, evid_dict = load_evidence(EVID_FILE)    


LABEL2ID = {"SUPPORTS": 0, "REFUTES": 1, "NOT_ENOUGH_INFO": 2, "DISPUTED": 3}
ID2LABEL = {v: k for k, v in LABEL2ID.items()}

for df in (df_train, df_dev):
    df["label_id"] = df["label"].map(LABEL2ID).astype("int8")

print(f"Train size: {len(df_train):,}")
print(f"Dev   size: {len(df_dev):,}")
print(f"Test  size: {len(df_test):,}")
print(f"Evidence passages: {len(df_evid):,}")

display(df_train.head())
display(df_evid.head())

print("Train label distribution:")
display(df_train["label"].value_counts())

print("Dev label distribution:")
display(df_dev["label"].value_counts())


Train size: 1,228
Dev   size: 154
Test  size: 153
Evidence passages: 1,208,827


,claim_id,claim_text,label,evid_ids,label_id
0,claim-1937,Not only is there no scientific evidence that ...,DISPUTED,"[evidence-442946, evidence-1194317, evidence-1...",3
1,claim-126,El Niño drove record highs in global temperatu...,REFUTES,"[evidence-338219, evidence-1127398]",1
2,claim-2510,"In 1946, PDO switched to a cool phase.",SUPPORTS,"[evidence-530063, evidence-984887]",0
3,claim-2021,Weather Channel co-founder John Coleman provid...,DISPUTED,"[evidence-1177431, evidence-782448, evidence-5...",3
4,claim-2449,"""January 2008 capped a 12 month period of glob...",NOT_ENOUGH_INFO,"[evidence-1010750, evidence-91661, evidence-72...",2


,evid_id,evid_text
0,evidence-0,"John Bennet Lawes, English entrepreneur and ag..."
1,evidence-1,Lindberg began his professional career at the ...
2,evidence-2,``Boston (Ladies of Cambridge)'' by Vampire We...
3,evidence-3,"Gerald Francis Goyer (born October 20, 1936) w..."
4,evidence-4,He detected abnormalities of oxytocinergic fun...


Train label distribution:


label
SUPPORTS           519
NOT_ENOUGH_INFO    386
REFUTES            199
DISPUTED           124
Name: count, dtype: int64

Dev label distribution:


label
SUPPORTS           68
NOT_ENOUGH_INFO    41
REFUTES            27
DISPUTED           18
Name: count, dtype: int64

In [32]:
# First, create a dictionary mapping evid_id to evid_text for faster lookup
evid_dict = dict(zip(df_evid['evid_id'], df_evid['evid_text']))

# Create a new column for evidence text
df_train['evid_text'] = df_train['evid_ids'].apply(lambda x: [evid_dict[evid_id] for evid_id in x])

# If you want to see the results
print(df_train[['claim_text', 'label', 'evid_text']].head())



                                          claim_text            label  \
0  Not only is there no scientific evidence that ...         DISPUTED   
1  El Niño drove record highs in global temperatu...          REFUTES   
2             In 1946, PDO switched to a cool phase.         SUPPORTS   
3  Weather Channel co-founder John Coleman provid...         DISPUTED   
4  "January 2008 capped a 12 month period of glob...  NOT_ENOUGH_INFO   

                                           evid_text  
0  [At very high concentrations (100 times atmosp...  
1  [While ‘climate change’ can be due to natural ...  
2  [There is evidence of reversals in the prevail...  
3  [There is no convincing scientific evidence th...  
4  [With average temperature +8.1 °C (47 °F)., Th...  


In [44]:
sample_df_train = df_train[['claim_text', 'label', 'evid_text']].head(20)
sample_df_train

,claim_text,label,evid_text
0,Not only is there no scientific evidence that ...,DISPUTED,[At very high concentrations (100 times atmosp...
1,El Niño drove record highs in global temperatu...,REFUTES,[While ‘climate change’ can be due to natural ...
2,"In 1946, PDO switched to a cool phase.",SUPPORTS,[There is evidence of reversals in the prevail...
3,Weather Channel co-founder John Coleman provid...,DISPUTED,[There is no convincing scientific evidence th...
4,"""January 2008 capped a 12 month period of glob...",NOT_ENOUGH_INFO,"[With average temperature +8.1 °C (47 °F)., Th..."
5,The last time the planet was even four degrees...,NOT_ENOUGH_INFO,"[Sea levels were high worldwide, and much of t..."
6,Tree-ring proxy reconstructions are reliable b...,DISPUTED,[The deviation of some tree ring proxy measure...
7,"Under the most ambitious scenarios, they found...",DISPUTED,[The sheet has been of recent concern because ...
8,An additional kick was supplied by an El Niño ...,NOT_ENOUGH_INFO,[An El Niño is associated with warm and very w...
9,When stomata-derived CO2 (red) is compared to ...,SUPPORTS,[One study using evidence from stomata of foss...


## Format Prompt ##

In [45]:
def format_prompt(row):
    claim = row['claim_text']
    evidence = row['evid_text']
    return f"Claim: {claim}\n" \
           f"Evidence: {evidence}\n" \
           f"Labels: [SUPPORTS, REFUTES, NOT_ENOUGH_INFO, DISPUTED]\n" \
           f"Task: Classify the provided claim into provided classes, using the provided evidences\n" \
           f"ONLY RESPONSE WITH ONE LABEL"

prompts = sample_df_train.apply(format_prompt, axis=1)
prompts

0     Claim: Not only is there no scientific evidenc...
1     Claim: El Niño drove record highs in global te...
2     Claim: In 1946, PDO switched to a cool phase.\...
3     Claim: Weather Channel co-founder John Coleman...
4     Claim: "January 2008 capped a 12 month period ...
5     Claim: The last time the planet was even four ...
6     Claim: Tree-ring proxy reconstructions are rel...
7     Claim: Under the most ambitious scenarios, the...
8     Claim: An additional kick was supplied by an E...
9     Claim: When stomata-derived CO2 (red) is compa...
10    Claim: 195 countries signed the 2015 Paris Agr...
11    Claim: However, there is a process of accretio...
12    Claim: Mars, Triton, Pluto and Jupiter all sho...
13    Claim: Venus doesn't have a runaway greenhouse...
14    Claim: The latest NOAA report is “a reminder t...
15    Claim: […] in fact this pattern is already eme...
16    Claim: Venus is not hot because of a runaway g...
17    Claim: Models and direct observations find

In [50]:
# Set model to evaluation mode
model.eval()

# Generate text function
def generate_text(prompt, max_length=1000):
    # Tokenize and move to GPU
    inputs = tokenizer(prompt, return_tensors="pt").to("cuda")
    
    # Generate with proper parameters
    with torch.no_grad():  # Disable gradient calculation for inference
        outputs = model.generate(
            **inputs,
            max_length=max_length,
            num_return_sequences=1,
            temperature=0.7,
            do_sample=True,
            pad_token_id=tokenizer.eos_token_id
        )
    
    return tokenizer.decode(outputs[0], skip_special_tokens=True)
# Test the model

def extract_label(response):
    # get the last word of the response
    index = -1
    split_response = response.split(" ")

    label = split_response[index].strip()
    return label

idx = 0
for prompt in prompts:

    response = generate_text(prompt)
    extracted_label = extract_label(response)
    print(f"Prompt: {prompt}\n")
    print(f"Predicted Label: {extracted_label}\n")
    print(f"Correct Label: {sample_df_train.iloc[idx]['label']}\n")
    print(f"--------------------------------")
    idx += 1


Prompt: Claim: Not only is there no scientific evidence that CO2 is a pollutant, higher CO2 concentrations actually help ecosystems support more plant and animal life.
Evidence: ['At very high concentrations (100 times atmospheric concentration, or greater), carbon dioxide can be toxic to animal life, so raising the concentration to 10,000 ppm (1%) or higher for several hours will eliminate pests such as whiteflies and spider mites in a greenhouse.', 'Plants can grow as much as 50 percent faster in concentrations of 1,000 ppm CO 2 when compared with ambient conditions, though this assumes no change in climate and no limitation on other nutrients.', 'Higher carbon dioxide concentrations will favourably affect plant growth and demand for water.']
Labels: [SUPPORTS, REFUTES, NOT_ENOUGH_INFO, DISPUTED]
Task: Classify the provided claim into provided classes, using the provided evidences
ONLY RESPONSE WITH ONE LABEL

Predicted Label: REFUTES

Correct Label: DISPUTED

-----------------------

KeyboardInterrupt: 